# Alive Prediction 模型选型与决策链路复核

本 notebook 只做第二阶段 alive/churn probability modeling 的展示复核，不训练模型、不物化特征、不保存预测明细。

## 0. 项目阶段说明

- 第一阶段数据清洗 v2 已冻结。
- 当前展示第二阶段 alive/churn probability modeling。
- 主干模型只输出 `churn_probability_H = P(die_H = 1)`。
- `value_at_risk` / `business_priority_score` 是模型外后处理，不进入主模型选择。
- 当前阶段结论：`probability_candidate_v1 = logistic_regression + frequency_decay_v1 + raw`。
- 当前业务可用结论：`business_usable_probability_baseline = true`。

## 运行环境与报告加载

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from alg.evaluation.alive_prediction_story import (
    build_final_decision_table,
    build_stage_summary_table,
    load_csv_if_exists,
    plot_calibration_v2_before_after,
    plot_cutoff_entity_count_trend,
    plot_cutoff_label_rate_trend,
    plot_demand_shape_label_rate,
    plot_feature_smd_top,
    plot_feature_stability_before_after,
    plot_model_probability_metrics,
    read_md_head,
    render_missing_files_report,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
display(build_stage_summary_table(PROJECT_ROOT))
display(build_final_decision_table(PROJECT_ROOT))
missing = render_missing_files_report(PROJECT_ROOT)
display(missing[~missing["exists"]])

## 1. 数据与任务口径

- Entity 粒度：`manufacturer_code × hospital_code × drug_code`，当前实现中的药品粒度主键为 `drug_group`。
- 主评估 scope：`recurring_only`。
- Horizons：H3 / H6 / H12。
- 标签：`die_H = H 窗口内无 purchase_event`。
- `one_shot_only` 只作 diagnostic，不参与主模型选择。

In [ ]:
trainability = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_small_models_expanded_train/trainability_report_by_horizon.csv")
if trainability is None:
    trainability = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_small_models/trainability_report_by_horizon.csv")
display(trainability if trainability is not None else pd.DataFrame([{"warning": "trainability report missing"}]))

if trainability is not None:
    cols = [c for c in trainability.columns if "row" in c or "entity" in c or "positive" in c or c in {"horizon", "scope"}]
    display(trainability[cols].head(20))

## 2. 指标体系说明

- Brier / LogLoss / ECE：概率质量。
- AUC / PR_AUC：区分能力，PR-AUC 必须结合 positive rate 解读。
- Precision / Lift / NDCG：TopK 辅助排序诊断。
- ECE 好不代表排序强；AUC 一般不代表业务不可用；Lift 用于判断是否超过 base rate。

## 3. 多模型 baseline 与模型家族比较

In [ ]:
small_metrics = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_small_models/model_metrics_by_scope.csv")
expanded_metrics = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_small_models_expanded_train/model_metrics_by_scope.csv")
display(small_metrics.head(20) if small_metrics is not None else pd.DataFrame([{"warning": "small model metrics missing"}]))
display(expanded_metrics.head(20) if expanded_metrics is not None else pd.DataFrame([{"warning": "expanded train metrics missing"}]))
plot_model_probability_metrics(expanded_metrics if expanded_metrics is not None else small_metrics, title="Small Model Probability Metrics")

CatBoost 早期排序/value 指标较强；但主模型收敛为纯概率模型后，不能使用 value-weighted metrics 选择主模型。后续概率候选主线收敛为 Logistic / XGBoost，其中 Logistic 更稳。

## 4. 时间漂移分析

In [ ]:
entity_trend = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_temporal_drift/cutoff_entity_count_trend.csv")
label_trend = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_temporal_drift/cutoff_label_rate_trend.csv")
feature_shift = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_temporal_drift/feature_distribution_shift.csv")
drift_summary = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_temporal_drift/drift_aware_metric_summary.csv")

display(entity_trend.head(20) if entity_trend is not None else pd.DataFrame([{"warning": "entity trend missing"}]))
display(label_trend.head(20) if label_trend is not None else pd.DataFrame([{"warning": "label trend missing"}]))
display(feature_shift.sort_values("standardized_mean_diff", key=lambda s: s.abs(), ascending=False).head(10) if feature_shift is not None and "standardized_mean_diff" in feature_shift else pd.DataFrame([{"warning": "feature shift missing"}]))

plot_cutoff_entity_count_trend(entity_trend)
plot_cutoff_label_rate_trend(label_trend)
plot_feature_smd_top(feature_shift)

guardrails = read_md_head(PROJECT_ROOT / "reports/alive_prediction_temporal_drift/metric_interpretation_guardrails.md", 1800)
display(Markdown(guardrails))

结论：2024 cohort 与训练期不同，late-2024 Precision/NDCG 不可直接解释为模型更强。后续必须使用 cutoff-aware 评估，并同时看 Lift、ECE、LogLoss、Brier。

## 4.1 2024 数据采集覆盖扩张解释

现有时间漂移不应只解释为终端行为漂移。2024 年 recurring entity count 快速增长，同时 label positive rate 在年初下降、下半年回升，更符合 data regime shift / coverage expansion：线上采购要求变严后，系统采集到更多医院采购订单，监控样本池发生扩张。

因此不建议直接删除旧数据；更合理的做法是保留 pre-2024 作为旧采集制度样本，将 2024 标注为 transition regime，后续等 2025/2026 标签窗口闭合并物化后，再做 post-expansion calibration / validation。

In [ ]:
regime_inflow = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_data_regime_shift_review/cutoff_entity_inflow_decomposition.csv")
regime_age = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_data_regime_shift_review/cutoff_label_rate_by_entity_age.csv")
regime_first_seen = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_data_regime_shift_review/cutoff_label_rate_by_first_seen_period.csv")
regime_shape = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_data_regime_shift_review/demand_shape_by_regime.csv")
regime_closure = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_data_regime_shift_review/label_window_closure_check.csv")

if regime_inflow is not None:
    display(regime_inflow[regime_inflow["cutoff_month"].between("2024-01", "2024-12")][[
        "cutoff_month",
        "recurring_entity_count",
        "new_entity_count",
        "returning_entity_count",
        "one_shot_to_recurring_count",
        "old_recurring_count",
        "share_new_entity",
        "share_returning_entity",
        "share_one_shot_to_recurring",
        "share_old_recurring",
    ]])
else:
    display(pd.DataFrame([{"提示": "data regime shift inflow report missing"}]))

if regime_first_seen is not None:
    display(regime_first_seen[
        regime_first_seen["cutoff_month"].between("2024-01", "2024-12")
    ].groupby(["horizon", "first_seen_period"], dropna=False)["positive_rate"].mean().reset_index())

if regime_shape is not None:
    display(regime_shape)

if regime_closure is not None:
    display(regime_closure[regime_closure["cutoff_month"].astype(str).str.startswith("2025")].head(36))

display(Markdown("""
### 中文解读

- 2024 recurring entity count 的增长不是单一来源：旧 recurring 延续占主，但 returning entity 与 one-shot 转 recurring 在扩张期也有贡献。
- 2024 年初 positive rate 下降，结合 first_seen/age 分层，更像新进入或短历史样本改变了样本池，而不是模型突然变强或变弱。
- 2024 下半年 positive rate 回升可以解释为 transition cohort 老化与 base rate 回归，但这只是观察性诊断。
- 2025 H12 不能作为回升证据：当前 purchase data 只到 2025-12，且 2025 feature/label cutoff 未物化。
- 不建议删除 pre-2024 数据；建议把 pre-2024 / transition_2024 / post-expansion candidate 分开评估。
"""))

## 5. 扩展训练窗口与 XGBoost

In [ ]:
window_cmp = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_small_models_expanded_train/training_window_comparison.csv")
xgb_family = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_small_models_expanded_train/model_family_comparison_by_horizon.csv")
display(window_cmp.head(30) if window_cmp is not None else pd.DataFrame([{"warning": "training window comparison missing"}]))
display(xgb_family.head(30) if xgb_family is not None else pd.DataFrame([{"warning": "model family comparison missing"}]))

2022-only 是 smoke test；expanded train 缓解样本不足，但时间漂移仍存在。XGBoost 作为 nonlinear challenger 保留，但没有稳定超过 Logistic。

## 6. Feature ablation 与概率候选收敛

In [ ]:
ablation = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_feature_ablation/feature_ablation_metrics.csv")
prob_candidates = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_probability_consolidation/probability_candidate_metrics.csv")
shortlist = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_probability_consolidation/probability_final_shortlist_v2.csv")
display(ablation.head(30) if ablation is not None else pd.DataFrame([{"warning": "feature ablation metrics missing"}]))
display(shortlist if shortlist is not None else pd.DataFrame([{"warning": "shortlist v2 missing"}]))
plot_model_probability_metrics(prob_candidates, title="Probability Candidate Consolidation")

`all_features` 不一定最好；`base_recency_frequency_only` 是早期低泄露基线；`value_features` 不进入主概率模型；Logistic 在概率指标上更稳。

## 7. Calibration v1 / rolling-origin / stabilization

In [ ]:
cal_v1 = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_calibration_v1/calibration_metrics_before_after.csv")
rolling = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_rolling_origin_v1/rolling_origin_metrics.csv")
stabilization = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_probability_stabilization/candidate_decision_update.csv")

if cal_v1 is not None:
    cal_cols = [c for c in [
        "model", "feature_set", "horizon", "calibration_method",
        "brier_score_delta_after_minus_before",
        "log_loss_delta_after_minus_before",
        "ece_delta_after_minus_before",
        "auc_delta_after_minus_before",
        "pr_auc_delta_after_minus_before",
    ] if c in cal_v1.columns]
    display(cal_v1[cal_cols].head(30).rename(columns={
        "model": "模型",
        "feature_set": "特征组",
        "horizon": "预测窗口H",
        "calibration_method": "校准方法",
        "brier_score_delta_after_minus_before": "Brier变化_after-before",
        "log_loss_delta_after_minus_before": "LogLoss变化_after-before",
        "ece_delta_after_minus_before": "ECE变化_after-before",
        "auc_delta_after_minus_before": "AUC变化_after-before",
        "pr_auc_delta_after_minus_before": "PR_AUC变化_after-before",
    }))
else:
    display(pd.DataFrame([{"提示": "calibration v1 指标缺失"}]))

if rolling is not None:
    rolling_summary = rolling[rolling.get("scope", "recurring_only").eq("recurring_only") if "scope" in rolling.columns else rolling.index == rolling.index].copy()
    if "aggregation_method" in rolling_summary.columns:
        rolling_summary = rolling_summary[rolling_summary["aggregation_method"].eq("macro_by_cutoff")]
    display(rolling_summary.groupby(["fold", "model", "feature_set", "calibration_method"], dropna=False)[
        ["brier_score", "log_loss", "ece", "auc", "pr_auc"]
    ].mean(numeric_only=True).reset_index().sort_values(["fold", "brier_score"]).rename(columns={
        "fold": "滚动验证折",
        "model": "模型",
        "feature_set": "特征组",
        "calibration_method": "概率版本",
        "brier_score": "Brier",
        "log_loss": "LogLoss",
        "ece": "ECE",
        "auc": "AUC",
        "pr_auc": "PR_AUC",
    }).head(40))
else:
    display(pd.DataFrame([{"提示": "rolling-origin 指标缺失"}]))

if stabilization is not None:
    display(stabilization.rename(columns={
        "model": "模型",
        "feature_set": "特征组",
        "calibration_method": "概率版本",
        "old_decision": "旧决策",
        "new_decision": "新决策",
        "reason": "原因",
    }))
else:
    display(pd.DataFrame([{"提示": "stabilization 决策缺失"}]))

display(Markdown("""
### 中文结论

- Calibration v1 比较了 raw / Platt / isotonic，但校准改善不稳定：有些 horizon ECE 下降，但 LogLoss 或分箱稳定性变差。
- Rolling-origin v1 的核心目的不是提高分数，而是验证跨时间切分稳定性；结果显示 Logistic 比 XGBoost 更稳，XGBoost 没有稳定超过 Logistic。
- 在 stabilization 阶段，候选状态从 `needs_rolling_origin_validation` 更新为 `needs_drift_and_feature_stability_review`，说明当时不能 promote 的主因是时间漂移与特征稳定性，而不是模型家族不足。
- 这一阶段的结论是：先做稳定特征重构，再考虑校准；不要继续换模型或调参。
"""))

v1 阶段 raw / Platt / isotonic 改善不一致；rolling-origin 显示 Logistic 稳定但不能直接 promote，主要问题是时间漂移与校准不稳定。

## 8. Feature stability v1 与 calibration v2

In [ ]:
stability_shift = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_feature_stability_v1/feature_distribution_shift_before_after.csv")
feature_metrics = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_feature_stability_v1/feature_set_comparison_by_fold.csv")
cal_v2 = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_calibration_v2/calibration_v2_metrics_by_fold.csv")
decision_v2 = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_calibration_v2/probability_candidate_v1_decision_v2.csv")

display(stability_shift.head(30) if stability_shift is not None else pd.DataFrame([{"warning": "feature stability shift missing"}]))
plot_feature_stability_before_after(stability_shift)
display(feature_metrics.head(30) if feature_metrics is not None else pd.DataFrame([{"warning": "feature set comparison missing"}]))
display(decision_v2 if decision_v2 is not None else pd.DataFrame([{"warning": "decision v2 missing"}]))
plot_calibration_v2_before_after(cal_v2)

pd.DataFrame([
    {"candidate": "old baseline raw", "brier": 0.1963, "log_loss": 0.5892, "ece": 0.1402},
    {"candidate": "logistic_regression + frequency_decay_v1 + raw", "brier": 0.1911, "log_loss": 0.5822, "ece": 0.1346},
])

normalized_recency_v1 与 frequency_decay_v1 降低漂移；calibration v2 中 raw 最优，最终 promote `logistic_regression + frequency_decay_v1 + raw`。

## 9. Demand-shape routing 与标签口径审查

In [ ]:
demand_dist = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_demand_shape_label_review/demand_shape_distribution.csv")
demand_label = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_demand_shape_label_review/demand_shape_label_rate_by_horizon.csv")
demand_metrics = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_demand_shape_label_review/demand_shape_probability_metrics.csv")
routing = load_csv_if_exists(PROJECT_ROOT / "reports/alive_prediction_demand_shape_label_review/demand_shape_routing_decision.csv")

display(demand_dist.head(30) if demand_dist is not None else pd.DataFrame([{"warning": "demand distribution missing"}]))
display(demand_label.head(30) if demand_label is not None else pd.DataFrame([{"warning": "demand label rate missing"}]))
plot_demand_shape_label_rate(demand_label)
display(demand_metrics.head(30) if demand_metrics is not None else pd.DataFrame([{"warning": "demand probability metrics missing"}]))
display(routing if routing is not None else pd.DataFrame([{"warning": "routing decision missing"}]))
display(Markdown(read_md_head(PROJECT_ROOT / "reports/alive_prediction_demand_shape_label_review/probability_candidate_v1_business_use_note.md", 2200)))
display(Markdown(read_md_head(PROJECT_ROOT / "reports/alive_prediction_demand_shape_label_review/next_stage_line_card_plan.md", 2200)))

需求形态结论：smooth 适合主概率模型；erratic 可用但 H3 谨慎；intermittent H12 优先、H3 observation-only；lumpy observation-only / longer horizon。当前标签总体可用，但低频需求形态需要 `label_confidence_weight`。

## 10. 最终结论

```text
probability_candidate_v1 = logistic_regression + frequency_decay_v1 + raw
business_usable_probability_baseline = true
```

可以说：
- 可作为初版流失概率基线；
- ECE 约 13.46%，可支持粗分层预警；
- 可进入线索卡/证据链原型；
- 可辅助人工复核。

不能说：
- 模型能精准区分每个终端；
- high risk 一定流失；
- low risk 一定安全；
- 可以自动派单；
- 可以替代人工判断。

下一步：线索卡/证据链原型、demand-shape guardrails、label_confidence_weight、真实客户流失名单回测。